# Federal + state tax filing: Databricks + Sigma demo

Connection: Sigma connection **exalabs** (`115b0210-65d6-4604-ac51-b135df09df41`), schema `exalabs.tax`.

| File | Purpose |
| --- | --- |
| `ddl.sql` | Schema, 6 dimensions, `fact_return`, 2 large facts (liquid clustering, informational PK/FK `RELY`), 2 gold views |
| `generate_tax_data.py` | Databricks notebook: 1.75M employees x 5 years, ~122.5M federal + ~181M state rows (`EMPLOYEES` constant to scale) |

Differences from the planning sheet (deliberate): added `fact_return` (employee x year, ~8.75M rows) so
"complete / in review / amended" is a cheap count rather than COUNT DISTINCT over ~300M rows; added `income_type_key`
and `load_ts` to the federal fact (needed for the Federal summary grain and the relative-date demo). Row counts are
generated, not measured: run the notebook's final `count()` cell and quote those.

## Run order
1. Databricks SQL: run `ddl.sql`.
2. Import `generate_tax_data.py`, attach multi-worker compute (smoke test with `EMPLOYEES = 50_000`), run all.
3. In Sigma, tell Claude the tables exist; it then builds the data model + workbook below on the exalabs connection.

## Built workbook
[Tax Filing Dashboard](https://staging.sigmacomputing.io/papercranestaging/workbook/Tax-Filing-Dashboard-4asDFNeSBw9bKXnxu711aA) on the exalabs connection.
Pages: Overview (Q1, Q2), Return status (Q3), Employee drill-down (Q4, default return `2025-1000`), Granular detail (demo 4,
defaults Tax year 2025 and entity Business Unit 001), Materialization targets (the `Federal summary` and `State summary` tables
to schedule for demo 1). Overview and Return status are live (the baseline); repoint them at the materialized
summaries after demo 1. Overview's effective-rate table and combined-rate KPI come from the data model's custom SQL element. Not built: row-level security and the Assistant. All workbook elements now read from the Tax Filing data model below (still live against Databricks, so the baseline for demo 1 is unchanged).

## Built data model
[Tax Filing](https://staging.sigmacomputing.io/papercranestaging/data-model/Tax-Filing-2YiEGHhwJA31MH8S9rRdw) (folder `IOT Apps Asssets`).
Elements: Federal detail, State detail, Returns, Effective rate by entity and year (custom SQL on Databricks, one row per
entity x year, the materialization candidate), plus the Entity, State, Tax line and Employee dimensions with relationships.
Metrics are ratios of sums: Federal tax / Gross income / Federal effective rate, State tax / Apportioned income / State
effective rate, Returns by status, and Combined effective rate (20.9% at last check; federal 17.9%, state 3.2% of apportioned
income). Tax line keys are hard-coded in the metrics (10 federal tax, 1-6 income lines, 106 apportioned, 109 state tax); a
line-code lookup would be cleaner. State share of total is left to the workbook (PercentOfTotal) because a metric version returned null per group.

## Sigma objects to build
**Data model `Tax Filing`**: elements `Federal detail` and `State detail` (relationships to dim_entity, dim_state,
dim_tax_year, dim_tax_line, dim_income_type, dim_employee, fact_return), plus `Federal summary`, `State summary`,
`Effective rate`. Metrics: Federal tax, State tax, Effective rate = (federal+state tax) / gross income (**ratio of sums,
never an average of rates**), State share of total, YoY change.
**Workbook `Tax Filing Dashboard`**: Overview (Q1, Q2), Return status (Q3), Employee drill-down (Q4), Granular detail (demo 4).

## Materialization decisions
| Element | Grain | Materialize? | Reasoning |
| --- | --- | --- | --- |
| Federal summary | entity x year x income type, ~60K | **Yes**, scheduled after the Delta load | Scans ~122.5M rows for a 60K-row answer; every Overview page reads it. Keep `entity_key` in the grain. |
| State summary | entity x state x year, ~90K | **Yes**, after the federal one | Reads ~181M rows; if it pulls federal AGI from the Federal summary, schedule it after its parent. |
| Effective-rate view | entity x year | **No (inherit)** | Tiny join of two materialized parents; its own copy adds a refresh to fail for no speed. |
| Granular detail views | line level | **No** | Dynamic filters/grouping mean no stable result to store, and ~300M rows is expensive to copy. Keep live, require filters, rely on clustering. |
| Row-security / user-attribute elements | any | **Materialize the unfiltered parent; apply the filter in a child** | One shared copy serves everyone. Verify in your tenant how user-attribute filters interact with materialization before the demo. |
| Relative-date filters ("last 30 days") | any | **Materialize without the relative filter; filter in a child** | A relative window moves every day, so a baked-in copy goes stale. Fits `load_ts`/`filed_date`, not `tax_year`. |

## Demo scripts
**1. Materialization scope.** (a) Baseline: Federal summary live, record duration in Query history. (b) Materialize Federal
summary; re-run, compare. (c) Materialize State summary; confirm refresh order. (d) Add a child with an entity row filter;
show it reads the parent's materialization. (e) Put the Entity control on the parent vs the child: parent target reruns on
the warehouse; child target stays on the materialized copy.
**2. Caching.** Run the workbook as a service account (same identity = shared cache). Show Databricks serverless/universal
result cache hits in Query history ("from cache"), then the workbook's cache duration setting; change per-workbook duration
and show the effect. Note that per-user credentials or RLS that changes the SQL text defeat the cache.
**3. Failure / monitoring.** Break the source (rename a column in a copy of `v_federal_summary`, or revoke the service
principal's SELECT), trigger a materialization, show the failure, queries falling back to live data, auto-pause after
repeated failures, then fix and resume from Admin > Materializations.
**4. Granular detail.** Table over `Federal detail` + `State detail` with required filters (tax year, entity), grouping by
state/line, columns from unused dimensions dropped to show join pruning (Query history SQL has fewer joins), prefetch
settings, then open the Databricks query profile: files pruned (liquid clustering on `tax_year, entity_key`) vs read.
**5. Where logic lives**: see below.

## Where logic lives
| Layer | Put here | Why |
| --- | --- | --- |
| **Databricks** | Federal/state calculations, state-starts-from-federal lineage (`federal_agi_ref`), return status, SCD2 employee history, liquid clustering, stats, PK/FK `RELY`, UC row filters for hard security, scheduled loads; optional gold views if the SQL must be shared with other BI | Heavy, stable, shared by every tool; one definition of tax truth |
| **Sigma data model** | Relationships, friendly names, metrics (effective rate, share, YoY), summary elements + materialization schedules, user-attribute entity security for the app, controls, folder/column security | Business-facing, changes often, reviewable by analysts, needs no warehouse deploy |
| **Sigma workbook** | Layout, controls, required-filter UX, drill paths, view-level grouping | Presentation only; keep no business math here |
Rule of thumb: calculate the tax in Databricks, define *how to read it* in Sigma.


## Data generation results (run 2026-11-10)

Run with `EMPLOYEES = 1_750_000`. 9 tables loaded in `exalabs.tax` (5 dimensions, 4 facts), about 314M rows total. Counts were checked with SQL and the main foreign keys have no orphan rows. The schema also contains the views `v_federal_summary` and `v_state_summary`, which this generator does not load.

| Table | Type | Rows | What it holds |
| --- | --- | --- | --- |
| `dim_state` | Dimension | 55 | 50 states plus DC and 4 territories, with an income-tax flag |
| `dim_tax_year` | Dimension | 10 | Tax years 2016–2025 and their filing deadlines |
| `dim_entity` | Dimension | 200 | Business units, with region and type |
| `dim_income_type` | Dimension | 40 | 8 named income types and 32 placeholder types |
| `dim_tax_line` | Dimension | 300 | 14 core federal lines, 16 core state lines, 270 unused schedule lines |
| `dim_employee` | Dimension (history-tracked) | 1,925,574 | 1,750,000 current employees and 175,574 earlier versions |
| `fact_return` | Fact | 8,750,000 | One row per employee per year (1.75M x 5 years) |
| `fact_federal_tax_detail` | Fact | 122,500,000 | 14 federal lines per return |
| `fact_state_tax_detail` | Fact | 181,128,080 | 16 lines for each of 11,320,505 state filings |

### Relationships
* `fact_return` links to `dim_employee` on `employee_key`, `dim_entity` on `entity_key` and `dim_tax_year` on `tax_year`. Its key is `return_id`, built as "year-employee_key".
* `fact_federal_tax_detail` links to `fact_return` on `return_id`. It also carries `employee_key`, `entity_key` and `tax_year` itself, and links to `dim_tax_line` on `tax_line_key`. Income lines (keys 1–6) also link to `dim_income_type`.
* `fact_state_tax_detail` has the same links plus `dim_state` on `state_key`. Its `federal_agi_ref` column repeats the federal AGI, so state amounts can be checked against the federal return.
* `dim_employee`: about 10% of employees have an older version whose key is the current key plus 1,750,000. That version points to a different entity and ended on 2022-12-31. The facts only use current keys (1–1,750,000).
* `dim_entity` and `dim_state` are also referenced from `dim_employee` (`entity_key`, `home_state_key`).

### State filings per return
Only 30% of employees get a second (work) state, and the generator drops it when it matches the home state. That gives about 1.29 state filings per return (11.32M filings for 8.75M returns), not 1.5. Raise `SECOND_STATE_PCT` for more state rows.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Tax filing synthetic data generator
# MAGIC Populates the tables created by `ddl.sql` (exalabs.tax). Defaults: 750K employees x 5 tax
# MAGIC years -> ~56M federal and ~90M state detail rows. Set EMPLOYEES=50_000 for a smoke test.
# MAGIC Run `ddl.sql` first. Re-running replaces all tables' contents (INSERT OVERWRITE).

# COMMAND ----------

from pyspark.sql import functions as F

CATALOG, SCHEMA = "exalabs", "tax"
EMPLOYEES = 1_750_000
TAX_YEARS = [2021, 2022, 2023, 2024, 2025]
ENTITIES = 200
SECOND_STATE_PCT = 0.30          # share of employees filing in a 2nd (work) state
T = lambda n: f"{CATALOG}.{SCHEMA}.{n}"

# COMMAND ----------
# MAGIC %md ## Dimensions

# COMMAND ----------

states = ("AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY "
          "NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC PR GU VI AS").split()
no_tax = {"AK", "FL", "NV", "NH", "SD", "TN", "TX", "WA", "WY", "GU", "VI", "AS"}
terr = {"PR", "GU", "VI", "AS"}
spark.createDataFrame(
    [(i + 1, c, f"State {c}", c in terr, c not in no_tax) for i, c in enumerate(states)],
    "state_key int, state_code string, state_name string, is_territory boolean, has_income_tax boolean",
).write.mode("overwrite").insertInto(T("dim_state"), overwrite=True)
assert len(states) == 55

spark.createDataFrame(
    [(y, __import__("datetime").date(y + 1, 4, 15), y == max(TAX_YEARS)) for y in range(2016, 2026)],
    "tax_year int, filing_deadline date, is_current_year boolean",
).write.mode("overwrite").insertInto(T("dim_tax_year"), overwrite=True)

regions = ["Northeast", "South", "Midwest", "West", "International"]
spark.range(1, ENTITIES + 1).select(
    F.col("id").cast("int").alias("entity_key"),
    F.format_string("E%03d", F.col("id")).alias("entity_code"),
    F.format_string("Business Unit %03d", F.col("id")).alias("entity_name"),
    F.array(*[F.lit(r) for r in regions])[(F.col("id") % 5).cast("int")].alias("region"),
    F.when(F.col("id") % 4 == 0, "Subsidiary").otherwise("Division").alias("entity_type"),
).write.mode("overwrite").insertInto(T("dim_entity"), overwrite=True)

named_income = ["Wages", "Bonus", "Equity (RSU/ESPP)", "Taxable interest", "Dividends",
                "Severance", "Relocation", "Fringe benefits"]
cats = ["Earned", "Earned", "Equity", "Investment", "Investment", "Other", "Other", "Other"]
inc = [(i + 1, n, cats[i]) for i, n in enumerate(named_income)]
inc += [(i + 1, f"Other income type {i + 1}", "Other") for i in range(len(inc), 40)]
spark.createDataFrame(inc, "income_type_key int, income_type_name string, income_category string") \
    .write.mode("overwrite").insertInto(T("dim_income_type"), overwrite=True)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Tax-line catalogue (300 rows)
# MAGIC Federal core lines 1-14, state core lines 101-116; the rest are defined-but-unused schedule lines
# MAGIC (realistic dimension size, exercises join pruning).

# COMMAND ----------

fed_core = [  # key, code, name, category, income_type_key
    (1, "F_WAGES", "W-2 wages", "INCOME", 1), (2, "F_BONUS", "Bonus", "INCOME", 2),
    (3, "F_EQUITY", "Equity compensation", "INCOME", 3), (4, "F_INT", "Taxable interest", "INCOME", 4),
    (5, "F_DIV", "Dividends", "INCOME", 5), (6, "F_OTHER", "Other income", "INCOME", 6),
    (7, "F_AGI", "Adjusted gross income", "ADJUSTMENT", None),
    (8, "F_STD_DED", "Standard deduction", "DEDUCTION", None),
    (9, "F_TAXABLE", "Taxable income", "TAXABLE", None),
    (10, "F_TAX", "Federal income tax", "TAX", None), (11, "F_CREDITS", "Credits", "CREDIT", None),
    (12, "F_WITHHELD", "Federal withholding", "PAYMENT", None),
    (13, "F_BAL", "Balance due / (refund)", "PAYMENT", None),
    (14, "F_EFF_RATE_BASIS", "Marginal bracket basis", "TAXABLE", None)]
st_core = [
    (101, "S_FED_AGI", "Federal AGI (starting point)", "INCOME"), (102, "S_ADD", "State additions", "ADJUSTMENT"),
    (103, "S_SUB", "State subtractions", "ADJUSTMENT"), (104, "S_AGI", "State AGI", "ADJUSTMENT"),
    (105, "S_ALLOC_PCT", "Allocation % to this state", "ADJUSTMENT"),
    (106, "S_APPORTIONED", "Apportioned income", "INCOME"), (107, "S_STD_DED", "State standard deduction", "DEDUCTION"),
    (108, "S_TAXABLE", "State taxable income", "TAXABLE"), (109, "S_TAX", "State income tax", "TAX"),
    (110, "S_LOCAL", "Local / city tax", "TAX"), (111, "S_SDI", "State disability insurance", "TAX"),
    (112, "S_PFL", "Paid family leave", "TAX"), (113, "S_CREDITS", "State credits", "CREDIT"),
    (114, "S_WITHHELD", "State withholding", "PAYMENT"), (115, "S_EST", "Estimated payments", "PAYMENT"),
    (116, "S_BAL", "Balance due / (refund)", "PAYMENT")]
lines = [(k, c, n, "FEDERAL", cat, it, k, True) for k, c, n, cat, it in fed_core]
lines += [(k, f"F_SCHED_{k}", f"Federal schedule line {k}", "FEDERAL", "ADJUSTMENT", None, k, False) for k in range(15, 101)]
lines += [(k, c, n, "STATE", cat, None, k - 100, True) for k, c, n, cat in st_core]
lines += [(k, f"S_SCHED_{k}", f"State schedule line {k}", "STATE", "ADJUSTMENT", None, k - 100, False) for k in range(117, 301)]
spark.createDataFrame(lines, "tax_line_key int, tax_line_code string, tax_line_name string, jurisdiction string, "
                      "line_category string, income_type_key int, line_sequence int, is_core_line boolean") \
    .write.mode("overwrite").insertInto(T("dim_tax_line"), overwrite=True)
assert len(lines) == 300

# COMMAND ----------
# MAGIC %md ## Employees (with a type-2 history row for ~10%)

# COMMAND ----------

h = lambda c, salt: F.abs(F.hash(c, F.lit(salt)).cast("long"))
base = spark.range(1, EMPLOYEES + 1).withColumnRenamed("id", "n")
emp_cur = base.select(
    F.col("n").alias("employee_key"),
    F.format_string("EMP%07d", "n").alias("employee_id"),
    F.format_string("Employee %d", "n").alias("employee_name"),
    (h("n", 1) % ENTITIES + 1).cast("int").alias("entity_key"),
    (h("n", 2) % 55 + 1).cast("int").alias("home_state_key"),
    F.when(h("n", 3) % 10 < 5, "Single").when(h("n", 3) % 10 < 9, "Married filing jointly")
     .otherwise("Head of household").alias("filing_status"),
    F.lit("2020-01-01").cast("date").alias("effective_from"),
    F.lit(None).cast("date").alias("effective_to"), F.lit(True).alias("is_current"))
# history: 10% changed entity; old version keeps a surrogate key above the current range
hist = emp_cur.where(h("employee_key", 9) % 10 == 0).select(
    (F.col("employee_key") + EMPLOYEES).alias("employee_key"), "employee_id", "employee_name",
    ((F.col("entity_key") % ENTITIES) + 1).alias("entity_key"), "home_state_key", "filing_status",
    "effective_from", F.lit("2022-12-31").cast("date").alias("effective_to"), F.lit(False).alias("is_current"))
emp_cur.unionByName(hist).write.mode("overwrite").insertInto(T("dim_employee"), overwrite=True)

# COMMAND ----------
# MAGIC %md ## Return-year spine -> fact_return, federal detail, state detail

# COMMAND ----------

yrs = spark.createDataFrame([(y,) for y in TAX_YEARS], "tax_year int")
spine = (emp_cur.select("employee_key", "entity_key", "home_state_key").crossJoin(yrs)
         .withColumn("return_id", F.concat_ws("-", F.col("tax_year"), F.col("employee_key")))
         .withColumn("salary", (40000 + h("employee_key", 4) % 260000) * F.pow(F.lit(1.04), F.col("tax_year") - 2021))
         .withColumn("r", h("employee_key", 5) + F.col("tax_year")))
load_ts = F.current_timestamp()

# fact_return: status mix; most recent year has more returns still in review
status = (F.when(F.col("r") % 100 < F.when(F.col("tax_year") == max(TAX_YEARS), 55).otherwise(86), "COMPLETE")
           .when(F.col("r") % 100 < F.when(F.col("tax_year") == max(TAX_YEARS), 90).otherwise(94), "IN_REVIEW")
           .otherwise("AMENDED"))
spine.select("return_id", "employee_key", "entity_key", "tax_year", status.alias("return_status"),
             F.expr("make_date(tax_year + 1, 1 + int(r % 4), 1 + int(r % 27))").alias("filed_date"),
             load_ts.alias("load_ts")).write.mode("overwrite").insertInto(T("fact_return"), overwrite=True)

# Federal calc. gross split across income types; deterministic amounts so state can reference AGI.
g = (spine
     .withColumn("wages", F.col("salary") * 0.80).withColumn("bonus", F.col("salary") * 0.08)
     .withColumn("equity", F.col("salary") * 0.06 * (h("employee_key", 6) % 3)).withColumn("interest", F.col("salary") * 0.01)
     .withColumn("div", F.col("salary") * 0.01).withColumn("other", F.col("salary") * 0.02)
     .withColumn("gross", F.expr("wages+bonus+equity+interest+div+other"))
     .withColumn("agi", F.col("gross") * 0.95)
     .withColumn("std", F.lit(13850.0) * F.pow(F.lit(1.03), F.col("tax_year") - 2022))
     .withColumn("taxable", F.greatest(F.col("agi") - F.col("std"), F.lit(0.0)))
     .withColumn("tax", F.col("taxable") * (0.10 + 0.27 * F.least(F.col("taxable") / 500000, F.lit(1.0))))
     .withColumn("credits", F.when(h("employee_key", 7) % 5 == 0, F.lit(2000.0)).otherwise(0.0))
     .withColumn("withheld", F.col("tax") * 1.03)
     .withColumn("bal", F.col("tax") - F.col("credits") - F.col("withheld"))
     .withColumn("basis", F.col("taxable")))

stack = ("stack(14, 1,1,wages, 2,2,bonus, 3,3,equity, 4,4,interest, 5,5,div, 6,6,other, "
         "7,cast(null as int),agi, 8,cast(null as int),std, 9,cast(null as int),taxable, 10,cast(null as int),tax, "
         "11,cast(null as int),credits, 12,cast(null as int),withheld, 13,cast(null as int),bal, 14,cast(null as int),basis)"
         " as (tax_line_key, income_type_key, amount)")
fed = g.selectExpr("return_id", "employee_key", "entity_key", "tax_year", stack)
fed = (fed.select("return_id", "employee_key", "entity_key", "tax_year", "tax_line_key", "income_type_key",
                  F.col("amount").cast("decimal(18,2)").alias("amount"), load_ts.alias("load_ts"))
       .repartition(400, "tax_year"))
fed.write.mode("overwrite").insertInto(T("fact_federal_tax_detail"), overwrite=True)

# State: home state gets (1 - alloc), optional work state gets alloc. Starts from federal AGI.
second = (h("employee_key", 8) % 100 < SECOND_STATE_PCT * 100)
alloc = F.when(second, F.lit(0.30)).otherwise(F.lit(0.0))
st_rows = (g.select("return_id", "employee_key", "entity_key", "tax_year", "agi", "home_state_key",
                    "salary", alloc.alias("alloc"),
                    ((h("employee_key", 10) % 55 + 1).cast("int")).alias("work_state_key")))
home = st_rows.withColumn("state_key", F.col("home_state_key")).withColumn("share", 1 - F.col("alloc"))
work = (st_rows.where(F.col("alloc") > 0).withColumn("state_key", F.col("work_state_key"))
        .withColumn("share", F.col("alloc")).where(F.col("state_key") != F.col("home_state_key")))
sx = home.unionByName(work)
rate = (F.lit(0.01) + (F.col("state_key") % 11) * 0.007)
sx = (sx.join(spark.table(T("dim_state")).select("state_key", "has_income_tax"), "state_key")
      .withColumn("add", F.col("agi") * 0.01).withColumn("sub", F.col("agi") * 0.02)
      .withColumn("s_agi", F.col("agi") + F.col("add") - F.col("sub"))
      .withColumn("apport", F.col("s_agi") * F.col("share"))
      .withColumn("sstd", F.lit(5000.0) * F.col("share"))
      .withColumn("staxable", F.greatest(F.col("apport") - F.col("sstd"), F.lit(0.0)))
      .withColumn("stax", F.when(F.col("has_income_tax"), F.col("staxable") * rate).otherwise(0.0))
      .withColumn("local", F.col("stax") * 0.05).withColumn("sdi", F.col("apport") * 0.001)
      .withColumn("pfl", F.col("apport") * 0.0008).withColumn("scred", F.lit(0.0))
      .withColumn("swh", F.col("stax") * 1.02).withColumn("est", F.lit(0.0))
      .withColumn("sbal", F.col("stax") - F.col("scred") - F.col("swh") - F.col("est")))
sstack = ("stack(16, 101,agi, 102,add, 103,sub, 104,s_agi, 105,share*100, 106,apport, 107,sstd, 108,staxable, "
          "109,stax, 110,local, 111,sdi, 112,pfl, 113,scred, 114,swh, 115,est, 116,sbal) as (tax_line_key, amount)")
state = (sx.selectExpr("return_id", "employee_key", "entity_key", "state_key", "tax_year", "agi", sstack)
         .select("return_id", "employee_key", "entity_key", "state_key", "tax_year", "tax_line_key",
                 F.col("amount").cast("decimal(18,2)").alias("amount"),
                 F.col("agi").cast("decimal(18,2)").alias("federal_agi_ref"), load_ts.alias("load_ts"))
         .repartition(400, "tax_year"))
state.write.mode("overwrite").insertInto(T("fact_state_tax_detail"), overwrite=True)

# COMMAND ----------
# MAGIC %md ## Optimize + stats (liquid clustering, column stats for join pruning / CBO)

# COMMAND ----------

for t in ["fact_federal_tax_detail", "fact_state_tax_detail", "fact_return", "dim_employee"]:
    spark.sql(f"OPTIMIZE {T(t)}")
for t in ["fact_federal_tax_detail", "fact_state_tax_detail", "fact_return", "dim_employee", "dim_entity",
          "dim_state", "dim_tax_year", "dim_income_type", "dim_tax_line"]:
    spark.sql(f"ANALYZE TABLE {T(t)} COMPUTE STATISTICS FOR ALL COLUMNS")
for t in ["fact_return", "fact_federal_tax_detail", "fact_state_tax_detail", "dim_employee"]:
    print(t, spark.table(T(t)).count())
